# Sumários da frota (Silver)

Tipagem dos contadores agregados de cada snapshot e conferência com as posições individuais do mesmo snapshot.

## Fluxo dos dados

* Origem: `cat_mobilidade_urbana_bronze.db_frota.tb_sumarios` e `cat_mobilidade_urbana_silver.db_frota.tb_posicoes_motoristas`
* Destino: `cat_mobilidade_urbana_silver.db_frota.tb_sumarios`

`qtdOnline` conta motoristas aptos a receber corrida, não veículos comprovadamente livres. Somar contadores entre snapshots não gera frota distinta.

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

In [0]:
sumarios = spark.table(f"{BRONZE}.tb_sumarios").select(
    F.col("codigo").alias("id_snapshot"),
    F.col("codigoRegiao").cast("int").alias("id_regiao"),
    F.to_timestamp("dataHora").alias("ts_snapshot"),
    *[
        F.col(origem).cast("int").alias(destino)
        for origem, destino in [
            ("qtdOnline", "qtd_online"),
            ("qtdSelecionado", "qtd_selecionado"),
            ("qtdACaminho", "qtd_a_caminho"),
            ("qtdComPassageiro", "qtd_com_passageiro"),
            ("qtdIndisponivel", "qtd_indisponivel"),
            ("qtdOffline", "qtd_offline"),
            ("qtdAproveitamento", "qtd_aproveitamento"),
        ]
    ],
).dropDuplicates(["id_snapshot"])

contagem_posicoes = (
    spark.table(f"{SILVER}.tb_posicoes_motoristas")
    .groupBy("id_snapshot")
    .agg(
        F.sum((F.col("estado") == "ONLINE").cast("int")).alias("pos_online"),
        F.sum((F.col("estado") == "SELECIONADO").cast("int")).alias("pos_selecionado"),
        F.sum((F.col("estado") == "A_CAMINHO").cast("int")).alias("pos_a_caminho"),
        F.sum((F.col("estado") == "COM_PASSAGEIRO").cast("int")).alias("pos_com_passageiro"),
        F.sum((F.col("estado") == "INDISPONIVEL").cast("int")).alias("pos_indisponivel"),
    )
)

resultado = (
    sumarios.join(contagem_posicoes, "id_snapshot", "left")
    .withColumn(
        "fl_consistente_com_posicoes",
        F.coalesce(
            (F.col("qtd_online") == F.col("pos_online"))
            & (F.col("qtd_selecionado") == F.col("pos_selecionado"))
            & (F.col("qtd_a_caminho") == F.col("pos_a_caminho"))
            & (F.col("qtd_com_passageiro") == F.col("pos_com_passageiro"))
            & (F.col("qtd_indisponivel") == F.col("pos_indisponivel")),
            F.lit(False),
        ),
    )
    .drop("pos_online", "pos_selecionado", "pos_a_caminho", "pos_com_passageiro", "pos_indisponivel")
    .withColumn("qtd_presentes", F.col("qtd_online") + F.col("qtd_selecionado") + F.col("qtd_a_caminho")
                + F.col("qtd_com_passageiro") + F.col("qtd_indisponivel"))
    .withColumn("data_local", F.to_date("ts_snapshot"))
    .withColumn("hora_local", F.hour("ts_snapshot"))
    .withColumn("_ts_processamento", F.current_timestamp())
)

gravar(
    resultado,
    f"{SILVER}.tb_sumarios",
    "Contadores tipados por snapshot da frota e conferencia com as posicoes individuais do mesmo snapshot. "
    "qtd_online nao prova veiculo livre.",
)

In [0]:
for linha in (
    spark.table(f"{SILVER}.tb_sumarios")
    .agg(F.count("*").alias("snapshots"), F.avg(F.col("fl_consistente_com_posicoes").cast("int")).alias("pct_consistente"))
    .collect()
):
    print(linha.asDict())